<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/07-convolutional-networks-vision-backbones.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **卷积神经网络与现代视觉骨干网络** {#convolutional-neural-networks-modern-vision-backbones}

第 06 章把架构看作泛化流程的一部分。本章深入讲解让深度视觉成为可能的架构假设：相邻像素会形成局部模式，同一种模式可以出现在许多位置，有用特征可以从边缘逐步组织为部件和对象。卷积神经网络（CNN）通过**局部连接**、**权重共享**和空间特征图层级编码这些假设。

本章继续使用第 06 章的 UCI/scikit-learn Digits 任务。复用固定的 60/20/20 划分后，主要变化因素就是架构。图像只有 $8\times8$，所以模型会刻意保持紧凑；若照搬将分辨率缩小 32 倍的 ImageNet stem，输入会被直接抹去。这本身就是一条架构原则：骨干网络必须尊重输入分辨率和任务几何。

数据来源：[scikit-learn `load_digits`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) 与 [UCI Optical Recognition of Handwritten Digits](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B)（CC BY 4.0，DOI `10.24432/C50P49`）。

### **为什么图像需要空间归纳偏置** {#why-images-need-spatial-inductive-bias}

一张具有 $C$ 个通道的 $H\times W$ 图像可以展平为长度 $CHW$ 的向量，但展平会丢掉“相邻坐标彼此相关”这一显式结构。从 $CHW$ 输入映射到 $D$ 输出的全连接层使用 $D(CHW+1)$ 个参数，并为每个位置分配不同权重。卷积只学习小型核并在所有位置复用。一个从 $C_{in}$ 到 $C_{out}$、大小为 $K_h\times K_w$ 的卷积使用

$$
C_{out}\left(C_{in}K_hK_w+1\right)
$$

个参数，与图像高度和宽度无关。

权重共享在远离边界时产生**平移等变性**：输入平移会使特征图对应平移。等变性不是不变性。只有经过池化、聚合、增强或学习得到的下游决策，分类器才会降低位置敏感性。Padding、stride、有限边界和绝对位置组件也会使等变性只是近似成立。

因此 CNN 并非普遍优于全连接网络。只有当局部性和重复模式符合数据结构时，这种偏置才有价值。对没有有意义邻域的表格特征，同一偏置可能有害。在数据和算力充足的图像任务上，Vision Transformer 可以凭借较弱的局部性假设学习更广泛的交互；但在样本效率、延迟或部署成熟度重要时，CNN 仍然很强。

<details>
<summary><strong>PyTorch：在同一 Digits 划分上比较全连接与卷积偏置</strong></summary>

```python
import math
import random
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=707):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
targets = torch.tensor(digits.target, dtype=torch.long)
indices = np.arange(len(targets))
dev_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=606, stratify=digits.target
)
train_idx, val_idx = train_test_split(
    dev_idx, test_size=0.25, random_state=606, stratify=digits.target[dev_idx]
)
x_train, y_train = images[train_idx], targets[train_idx]
x_val, y_val = images[val_idx], targets[val_idx]
x_test, y_test = images[test_idx], targets[test_idx]


def loader(x, y, shuffle=False, seed=707, batch_size=128):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=shuffle, generator=generator)


@torch.no_grad()
def evaluate(model, x=x_val, y=y_val):
    model.eval()
    logits = model(x)
    return {"loss": F.cross_entropy(logits, y).item(),
            "accuracy": (logits.argmax(1) == y).float().mean().item()}


def fit(model, epochs=25, lr=3e-3, seed=707):
    seed_everything(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader(x_train, y_train, shuffle=True, seed=seed):
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
    return evaluate(model)


class DenseDigits(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.net(x)


class CompactCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(32, 10)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


seed_everything(707)
dense = DenseDigits()
dense_result = fit(dense, seed=707)
seed_everything(707)
cnn = CompactCNN()
cnn_result = fit(cnn, seed=707)

parameter_count = lambda model: sum(p.numel() for p in model.parameters())
assert cnn(x_train[:8]).shape == (8, 10)
assert 0 <= cnn_result["accuracy"] <= 1
print({"dense": (parameter_count(dense), dense_result),
       "cnn": (parameter_count(cnn), cnn_result)})
```

</details>

一次运行不是普遍基准。它建立了本章的实验契约，并展示如何在相同数据上比较参数化和留出表现。后续示例会拆解产生 CNN 偏置的具体机制。

### **离散卷积与互相关** {#discrete-convolution-cross-correlation}

深度学习库中名为 convolution 的操作通常实现的是**互相关（cross-correlation）**。对于输入 $X\in\mathbb{R}^{C_{in}\times H\times W}$ 和卷积核 $K\in\mathbb{R}^{C_{out}\times C_{in}\times K_h\times K_w}$，

$$
Y_{o,i,j}=b_o+
\sum_{c=1}^{C_{in}}
\sum_{u=0}^{K_h-1}
\sum_{v=0}^{K_w-1}
K_{o,c,u,v}\,X_{c,i+u,j+v}.
$$

数学卷积会在两个空间维度翻转卷积核，互相关不会。因为核参数由学习得到，两种约定经过重新参数化可以表示同一类滤波器；但在导入固定解析核或与信号处理公式对比时，这个差异很重要。

在每个输出位置，卷积核计算权重与局部 patch 的内积。较大的正响应表示局部模式与核方向一致，负响应表示相反对比，接近零表示匹配较弱。一个输出通道对应一个学习到的检测器，但它未必能被人类直接命名为某种边缘。堆叠多层后，后部滤波器可以组合早期局部响应。

![带 padding 的输入、移动卷积核和输出特征图展示了局部内积如何跨位置复用。](assets/dl07-convolution-padding.svg){fig-align="center" width="74%" fig-alt="卷积核在带 padding 的图像上移动并生成特征图的示意图。"}

*图片来源：依据上面的离散互相关公式绘制的本地教学图。*

<details>
<summary><strong>PyTorch：用真实 digit 的局部 patch 复现 `conv2d`</strong></summary>

```python
digit = x_train[:1]
edge_kernel = torch.tensor([[[[-1.0, 0.0, 1.0],
                              [-1.0, 0.0, 1.0],
                              [-1.0, 0.0, 1.0]]]])

library_response = F.conv2d(digit, edge_kernel, padding=1)
padded = F.pad(digit, (1, 1, 1, 1))
manual_response = torch.empty_like(library_response)
for row in range(8):
    for column in range(8):
        patch = padded[0, 0, row:row + 3, column:column + 3]
        manual_response[0, 0, row, column] = (patch * edge_kernel[0, 0]).sum()

flipped_response = F.conv2d(digit, edge_kernel.flip(-1, -2), padding=1)
assert torch.allclose(manual_response, library_response)
assert not torch.allclose(flipped_response, library_response)
print("strongest vertical-edge response:", library_response.abs().max().item())
```

</details>

手写循环适合理解语义，生产内核则会把卷积转化为优化过的分块运算。即使硬件实现并非真的逐 patch 滑动，数学输出仍是一组共享局部内积。

### **通道、卷积核、步幅、填充与空洞** {#channels-kernels-stride-padding-dilation}

在一个空间维度上，输入大小 $H$、核大小 $K$、dilation $D$、padding $P$ 和 stride $S$ 产生

$$
H_{out}=\left\lfloor
\frac{H+2P-D(K-1)-1}{S}+1
\right\rfloor.
$$

有效核跨度是 $K_{eff}=D(K-1)+1$。宽度使用相同公式。输出通道数由卷积核数量决定，而不是由空间公式决定。

- **Stride** 让卷积核一次前进多个像素，相当于滤波与下采样同时发生；若缺少适当低通行为，高频信息会混叠。
- **Padding** 决定边界覆盖和输出大小。零填充引入人工边框，反射或复制填充编码不同的边界假设。
- **Dilation** 拉开核采样点，在不增加参数的情况下扩大覆盖范围，但反复使用大 dilation 可能产生棋盘式盲区。
- **Channels** 在标准卷积中会被混合：每个输出通道汇总所有输入通道。分组卷积和 depthwise convolution 会有意限制这种混合。

PyTorch 的张量布局是 `[B, C, H, W]`。若把 channels-last 图像 `[B, H, W, C]` 当成该布局，会立即报错，或更危险地创建配置错误却仍可运行的模型。

<details>
<summary><strong>PyTorch：预测并验证特征图形状</strong></summary>

```python
batch = x_train[:12]
configurations = [
    {"kernel_size": 3, "stride": 1, "padding": 0, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "padding": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 2, "padding": 1, "dilation": 1},
    {"kernel_size": 3, "stride": 1, "padding": 2, "dilation": 2},
]

shape_records = []
for cfg in configurations:
    layer = nn.Conv2d(1, 6, bias=False, **cfg)
    output = layer(batch)
    k, s, p, d = cfg["kernel_size"], cfg["stride"], cfg["padding"], cfg["dilation"]
    expected = math.floor((8 + 2 * p - d * (k - 1) - 1) / s + 1)
    assert output.shape == (12, 6, expected, expected)
    shape_records.append((cfg, tuple(output.shape)))

print(shape_records)
```

</details>

形状计算不是架构完成后的记账工作，而是设计的一部分。每次降采样都会同时改变内存、计算量、感受野以及密集预测头可用的空间粒度。

### **感受野与特征层级** {#receptive-fields-feature-hierarchies}

一个单元的**理论感受野**是能够影响它的输入区域。令第 $l$ 层的感受野大小为 $r_l$，相邻单元在原始输入上的间隔为 $j_l$，则

$$
j_l=j_{l-1}S_l,
\qquad
r_l=r_{l-1}+(K_l-1)D_lj_{l-1},
$$

初始值 $r_0=j_0=1$。两个 stride-one 的 $3\times3$ 卷积得到 $5\times5$ 感受野，同时通常比一个稠密 $5\times5$ 卷积参数更少，并在中间增加一次非线性。stride-two 层会增大 jump，因此后续每个核都能更快扩大覆盖范围。

**有效感受野**通常更小且集中在中心，因为不同路径的梯度贡献不相等。它依赖学习权重、非线性门、归一化和数据。因此理论感受野覆盖整幅图像，并不能证明模型真正强烈使用了远处上下文。

特征层级是上下文逐步扩大的功能结果：早期层可检测对比和局部方向，中层组合局部图案，深层表示任务相关的部件和配置。这种解释有助于理解，但不能保证每个通道都有清晰语义；给特征图命名前需要表示探测和归因分析。

<details>
<summary><strong>PyTorch：通过输入梯度揭示感受野</strong></summary>

```python
probe = x_train[0:1].clone().requires_grad_(True)
stack = nn.Sequential(
    nn.Conv2d(1, 1, 3, padding=1, bias=False), nn.ReLU(),
    nn.Conv2d(1, 1, 3, padding=1, bias=False),
)
with torch.no_grad():
    for layer in stack:
        if isinstance(layer, nn.Conv2d):
            layer.weight.fill_(1.0)

feature = stack(probe)
feature[0, 0, 4, 4].backward()
support = probe.grad[0, 0].abs() > 0
rows, columns = support.nonzero(as_tuple=True)

height = int(rows.max() - rows.min() + 1)
width = int(columns.max() - columns.min() + 1)
assert (height, width) == (5, 5)
assert support.sum() == 25
print({"gradient support": (height, width), "nonzero pixels": int(support.sum())})
```

</details>

这里固定正权重，避免 ReLU 因随机符号关闭路径。对训练好的 CNN，应可视化梯度幅度而不仅是非零支持，从而观察有效感受野和边界伪影。

### **池化与分辨率变化** {#pooling-resolution-changes}

池化在局部邻域内聚合信息，但不会学习完整的通道混合核。最大池化保留最强激活：

$$
y_{c,i,j}=\max_{(u,v)\in\mathcal{N}_{i,j}}x_{c,u,v},
$$

平均池化则保留局部均值。当“特征是否存在”比精确位置更重要时，最大池化很有用，但梯度只经过获胜元素，并可能放大孤立噪声。平均池化会分散梯度，却可能模糊稀疏的判别响应。

降低分辨率带来三个好处：扩大后续感受野、降低激活内存并减少计算。代价是不可逆的空间信息损失。现代网络常用步幅卷积，让下采样滤波器可学习。检测与分割会保留多个分辨率，因为单一低分辨率图不足以处理小物体和精确边界。

全局平均池化把 `[B,C,H,W]` 映射为 `[B,C]`。相比展平后连接大规模全连接层，它减少参数，并鼓励每个通道成为图像级证据图。但它也丢弃显式布局，因此不适用于输出本身必须保持空间结构的任务。

<details>
<summary><strong>PyTorch：比较池化中的信息与梯度路由</strong></summary>

```python
pool_input = x_train[:4].clone().requires_grad_(True)
max_output = F.max_pool2d(pool_input, kernel_size=2, stride=2)
max_output.sum().backward(retain_graph=True)
max_nonzero_gradients = int((pool_input.grad != 0).sum())

pool_input.grad.zero_()
average_output = F.avg_pool2d(pool_input, kernel_size=2, stride=2)
average_output.sum().backward()
average_nonzero_gradients = int((pool_input.grad != 0).sum())

assert max_output.shape == average_output.shape == (4, 1, 4, 4)
assert max_nonzero_gradients <= 4 * 1 * 4 * 4
assert average_nonzero_gradients >= max_nonzero_gradients
print({"max gradient locations": max_nonzero_gradients,
       "average gradient locations": average_nonzero_gradients})
```

</details>

池化既不是必需的，也不是无害的。应根据输出几何和内存预算选择分辨率日程，并检验小特征表现，而不是假设下采样必然产生有用不变性。

### **深度卷积与可分离卷积** {#depthwise-separable-convolutions}

标准卷积同时混合空间与通道。对于 $C_{in}$ 个输入通道、$C_{out}$ 个输出通道和 $K\times K$ 核，其权重数量和每位置乘加量都与

$$
K^2C_{in}C_{out}
$$

成正比。**深度可分离卷积（depthwise separable convolution）**把它分解为：

1. depthwise $K\times K$ 卷积，每个输入通道使用一个空间滤波器，成本 $K^2C_{in}$；
2. pointwise $1\times1$ 卷积混合通道，成本 $C_{in}C_{out}$。

相对标准卷积的比例为

$$
\frac{K^2C_{in}+C_{in}C_{out}}{K^2C_{in}C_{out}}
=\frac{1}{C_{out}}+\frac{1}{K^2}.
$$

这种大幅算术削减支撑了 MobileNet 类模型，但它同时是一种结构约束：空间滤波不能立即使用任意跨通道组合。实际硬件加速也可能小于 FLOP 降幅，因为内存移动、内核启动开销和实现质量同样重要。

![Depthwise convolution 先为每个输入通道分配独立空间核，再进行 pointwise 通道混合。](assets/dl07-depthwise-separable.svg){fig-align="center" width="72%" fig-alt="先独立执行空间滤波、再进行通道混合的深度可分离卷积示意图。"}

*图片来源：在检索并参考 [Dive into Deep Learning Compiler, Depthwise Convolution](https://tvm.d2l.ai/chapter_common_operators/depthwise_conv.html) 及上方参数分解后绘制的本地教学图。*

<details>
<summary><strong>PyTorch：在 Digits 特征图上比较分解效果</strong></summary>

```python
features = nn.Conv2d(1, 32, 3, padding=1)(x_train[:16])
standard = nn.Conv2d(32, 64, 3, padding=1, bias=False)
separable = nn.Sequential(
    nn.Conv2d(32, 32, 3, padding=1, groups=32, bias=False),
    nn.Conv2d(32, 64, 1, bias=False),
)

standard_output = standard(features)
separable_output = separable(features)
standard_parameters = sum(p.numel() for p in standard.parameters())
separable_parameters = sum(p.numel() for p in separable.parameters())
theoretical = 3 * 3 * 32 + 32 * 64

assert standard_output.shape == separable_output.shape == (16, 64, 8, 8)
assert separable_parameters == theoretical
assert separable_parameters < standard_parameters
print({"standard": standard_parameters, "depthwise_separable": separable_parameters,
       "parameter ratio": separable_parameters / standard_parameters})
```

</details>

两种输出形状相同，但数值并不等价。分解定义了不同的假设空间；选择它是因为效率-准确率权衡更好，而不是因为它对任意标准卷积核都是代数恒等式。

### **从 LeNet、AlexNet 到 VGG** {#lenet-alexnet-vgg}

理解历史架构的最好方式，是观察它们如何改变**设计词汇**，而不是背诵层数。

- **LeNet-5** 为小型字符图像建立了卷积-池化-分类器模式，用局部感受野和共享权重代替对每个像素位置的全连接处理。
- **AlexNet** 证明更深 CNN、ReLU、Dropout、数据增强和 GPU 训练可以扩展图像识别。早期大卷积核和稠密分类头反映了当时的硬件与数据条件。
- **VGG** 通过反复堆叠 $3\times3$ 卷积，并在降低分辨率后加倍通道数，把深度设计系统化。两个 $3\times3$ 层获得 $5\times5$ 感受野，含两个非线性，且通常比单个稠密 $5\times5$ 层权重更少。

这些 ImageNet 架构不能原样复制到 $8\times8$ Digits。反复池化会使空间维度坍缩，大型全连接头会支配参数量。可复用的原则是 stage 模式：在一个 stage 内保持分辨率，然后用空间尺寸换取通道容量。

<details>
<summary><strong>PyTorch：把三种历史设计模式适配到 $8\times8$ 输入</strong></summary>

```python
class TinyLeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 3, padding=1), nn.Tanh(), nn.AvgPool2d(2),
            nn.Conv2d(6, 16, 3, padding=1), nn.Tanh(), nn.AvgPool2d(2),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 2 * 2, 32), nn.Tanh(), nn.Linear(32, 10))

    def forward(self, x):
        return self.head(self.features(x))


class TinyAlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 24, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(48, 10))

    def forward(self, x):
        return self.head(self.features(x))


class TinyVGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(32, 10)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


historical = [TinyLeNet(), TinyAlexNet(), TinyVGG()]
records = []
for model in historical:
    output = model(x_train[:8])
    records.append((model.__class__.__name__, sum(p.numel() for p in model.parameters()), tuple(output.shape)))
    assert output.shape == (8, 10)
print(records)
```

</details>

这些类保留了代表性模块，而不是复现论文基准。忠实复现还需要原始输入尺寸、预处理、初始化、优化器和训练配方；仅使用架构名称不会重现历史结果。

### **残差网络** {#residual-networks}

普通网络加深时，即使深层模型理论上可以表示浅层模型，优化也可能退化。残差模块围绕恒等路径学习残差函数 $F$：

$$
h_{l+1}=h_l+F(h_l;\theta_l).
$$

反向信号包含一个直接项：

$$
\frac{\partial L}{\partial h_l}
=\frac{\partial L}{\partial h_{l+1}}
\left(I+\frac{\partial F}{\partial h_l}\right),
$$

因此梯度不必完全穿过每个变换。这不能保证条件数完美，但让接近恒等的行为容易表达，并支持更深网络的优化。

相加要求形状一致。分辨率或通道数改变时，可用 stride-two 的 $1\times1$ 投影快捷支路对齐。归一化和激活的位置也很重要。预激活 ResNet 把归一化和激活放在卷积前，从而保留更干净的恒等路径。

![残差单元让输入同时经过恒等快捷路径和学习到的残差分支，再通过加法合并。](assets/dl07-residual-unit.svg){fig-align="center" width="68%" fig-alt="恒等快捷路径与学习分支通过加法合并的残差模块图。"}

*图片来源：依据残差映射公式与 [He et al., Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385) 绘制的本地教学图。*

<details>
<summary><strong>PyTorch：在 Digits 上训练紧凑残差网络</strong></summary>

```python
class BasicResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return F.relu(x + self.branch(x))


class TinyResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 24, 3, padding=1), nn.ReLU())
        self.blocks = nn.Sequential(BasicResidualBlock(24), BasicResidualBlock(24), BasicResidualBlock(24))
        self.head = nn.Linear(24, 10)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.head(x.mean((-2, -1)))


seed_everything(708)
resnet = TinyResNet()
resnet_validation = fit(resnet, epochs=25, lr=2e-3, seed=708)
resnet_test = evaluate(resnet, x_test, y_test)
assert resnet(x_train[:5]).shape == (5, 10)
assert resnet_validation["accuracy"] > 0.80
print({"validation": resnet_validation, "test": resnet_test})
```

</details>

残差连接解决的是优化接口问题，它不能替代数据质量、正则化或合适的分辨率日程。残差网络训练失败时，应先检查激活/梯度统计、归一化模式和投影形状，而不是直接把原因归为深度。

### **EfficientNet 与 ConvNeXt** {#efficientnet-convnext}

EfficientNet 和 ConvNeXt 代表通向强现代 CNN 的两条不同路线。

**EfficientNet** 从高效移动端模块出发，协同缩放深度、宽度和输入分辨率。简化的 compound scaling 规则是

$$
d=\alpha^\phi,\qquad w=\beta^\phi,\qquad r=\gamma^\phi,
\qquad \alpha\beta^2\gamma^2\approx2,
$$

其中 $\phi$ 是全局缩放系数。指数反映近似卷积成本：宽度同时影响输入和输出通道，分辨率影响两个空间轴。EfficientNet 模块通常组合 expansion、depthwise convolution、squeeze-and-excitation、projection 和残差路径。

**ConvNeXt** 在仍保持卷积的情况下，借鉴 Transformer 的设计选择更新 ResNet：patch 风格下采样 stem、大核 depthwise convolution、更少的激活/归一化位置、LayerNorm、inverted bottleneck、GELU，以及向后部倾斜的 stage 比例。它说明不是某个算子单独获胜；训练配方和系统级模块设计能让成熟归纳偏置保持竞争力。

FLOPs 只是延迟代理。Depthwise 层可能受内存带宽限制，更大激活会增加内存流量，理论高效的模块也可能不适配目标加速器。必须在部署 batch size 和精度下测量吞吐量、峰值内存和尾延迟。

<details>
<summary><strong>PyTorch：在 Digits 特征上比较移动端与 ConvNeXt 风格模块</strong></summary>

```python
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.SiLU(),
            nn.Conv2d(hidden, channels, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.net(x)


class MBConv(nn.Module):
    def __init__(self, channels=24, expansion=4):
        super().__init__()
        hidden = channels * expansion
        self.branch = nn.Sequential(
            nn.Conv2d(channels, hidden, 1), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden), nn.SiLU(),
            SqueezeExcitation(hidden), nn.Conv2d(hidden, channels, 1),
        )

    def forward(self, x):
        return x + self.branch(x)


class ConvNeXtBlock(nn.Module):
    def __init__(self, channels=24, expansion=4):
        super().__init__()
        self.depthwise = nn.Conv2d(channels, channels, 7, padding=3, groups=channels)
        self.norm = nn.LayerNorm(channels)
        self.expand = nn.Linear(channels, channels * expansion)
        self.contract = nn.Linear(channels * expansion, channels)

    def forward(self, x):
        residual = x
        x = self.depthwise(x).permute(0, 2, 3, 1)
        x = self.contract(F.gelu(self.expand(self.norm(x))))
        return residual + x.permute(0, 3, 1, 2)


stem_features = nn.Conv2d(1, 24, 3, padding=1)(x_train[:16])
mobile_block, convnext_block = MBConv(), ConvNeXtBlock()
mobile_output = mobile_block(stem_features)
convnext_output = convnext_block(stem_features)

count = lambda module: sum(p.numel() for p in module.parameters())
assert mobile_output.shape == convnext_output.shape == stem_features.shape
print({"MBConv parameters": count(mobile_block),
       "ConvNeXt-style parameters": count(convnext_block)})
```

</details>

这些是组件级实现，不是完整 EfficientNet 或 ConvNeXt 复现。它们暴露数据路径和张量布局，同时把大规模训练配方和优化内核交给 torchvision 等成熟库。

### **U-Net、特征金字塔、检测与分割** {#unet-feature-pyramids-detection-segmentation}

分类把图像映射到一个标签；检测和分割必须保留或重建空间对应关系。反复下采样的骨干网络会获得语义和上下文，却丢失边界细节。

**U-Net** 使用编码器-解码器解决这一问题。编码器生成分辨率逐步降低的特征，解码器将其上采样。Skip connection 在匹配分辨率上拼接编码器特征，使解码器同时获得语义上下文和细节。拼接不同于残差相加：它把两组张量保留为独立通道，让后续卷积决定如何组合。

**Feature Pyramid Network（FPN）**通过自顶向下路径和横向 $1\times1$ 投影，在多个分辨率上构造语义较强的特征图。检测头可让高分辨率层处理小物体，让低分辨率层处理大物体。核心差异在任务接口：U-Net 通常解码一个密集输出，而 FPN 向任务头暴露可复用的多尺度表示。

检测还需要预测类别和位置。Anchor-based 头对预定义边界框打分并回归偏移；anchor-free 头直接预测中心、角点或距离。分割则输出逐像素类别分布。它们的损失、分配规则和评估指标不同，但都依赖保留空间层级。

Digits 标签不包含分割 mask。为了做机制级实验，本章把大于零的像素定义为确定性的**前景代理**。这不是研究级分割基准，而是让同一批源图像能够验证 skip shape 和密集输出，而不引入另一数据集。

<details>
<summary><strong>PyTorch：为 Digits 前景 mask 构造微型 U-Net 路径</strong></summary>

```python
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(8, 16, 3, padding=1), nn.ReLU())
        self.decode = nn.Sequential(nn.Conv2d(24, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 1, 1))

    def forward(self, x):
        high_resolution = self.enc1(x)                         # [B, 8, 8, 8]
        low_resolution = self.enc2(F.max_pool2d(high_resolution, 2))  # [B, 16, 4, 4]
        upsampled = F.interpolate(low_resolution, size=high_resolution.shape[-2:], mode="bilinear", align_corners=False)
        logits = self.decode(torch.cat([high_resolution, upsampled], dim=1))
        return logits, (high_resolution, low_resolution, upsampled)


segmenter = TinyUNet()
digit_batch = x_train[:32]
foreground = (digit_batch > 0).float()
mask_logits, pyramid = segmenter(digit_batch)
mask_loss = F.binary_cross_entropy_with_logits(mask_logits, foreground)
mask_loss.backward()

assert mask_logits.shape == foreground.shape == (32, 1, 8, 8)
assert [tuple(t.shape[-2:]) for t in pyramid] == [(8, 8), (4, 4), (8, 8)]
assert all(parameter.grad is not None for parameter in segmenter.parameters())
print({"mask loss": mask_loss.item(), "feature shapes": [tuple(t.shape) for t in pyramid]})
```

</details>

该示例验证拓扑而非分割质量。有效应用需要人工或业务流程生成的 mask，按实体划分数据，处理类别不平衡，并使用 IoU 或 Dice 等反映空间重叠的指标。

### **CNN 与 Vision Transformer** {#cnns-vs-vision-transformers}

CNN 使用共享卷积核聚合局部邻域。Vision Transformer（ViT）把图像分为 patch，将每个 patch 嵌入为 token，加入位置信息，再用 self-attention 让每个 token 与其他 token 交互。

对 patch 大小 $P$，一张 $H\times W$ 图像会得到

$$
N=\frac{H}{P}\frac{W}{P}
$$

个 token。完整 self-attention 的交互成本为 $O(N^2D)$，其中 $D$ 是嵌入维度；固定通道和核大小的卷积对空间位置呈线性成本。ViT 的全局交互很灵活，但通常更依赖数据规模、预训练、增强和正则化。混合式和分层 Transformer 会重新引入局部性或多尺度结构，因为稠密全局 attention 并不是唯一有用的图像先验。

比较必须包括训练配方和部署环境。预训练 ViT 可能比从头训练的 CNN 迁移更好；紧凑 CNN 可能在 batch size 1 时更快；窗口 attention 的扩展规律也不同于全局 attention。“CNN 对 Transformer”并不是单个算子的对决。

<details>
<summary><strong>PyTorch：把同一批 $8\times8$ digits 转换为图像 token</strong></summary>

```python
class TinyVisionTransformer(nn.Module):
    def __init__(self, patch_size=2, dimension=32, heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.patch_projection = nn.Linear(patch_size * patch_size, dimension)
        self.class_token = nn.Parameter(torch.zeros(1, 1, dimension))
        self.position = nn.Parameter(torch.zeros(1, 17, dimension))  # 16 patches + class token
        layer = nn.TransformerEncoderLayer(
            d_model=dimension, nhead=heads, dim_feedforward=64,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Linear(dimension, 10)

    def forward(self, x):
        patches = F.unfold(x, kernel_size=self.patch_size, stride=self.patch_size).transpose(1, 2)
        tokens = self.patch_projection(patches)
        cls = self.class_token.expand(x.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.position[:, :tokens.shape[1] + 1]
        encoded = self.encoder(tokens)
        return self.head(encoded[:, 0]), patches


vit = TinyVisionTransformer()
vit_logits, digit_patches = vit(x_train[:8])
cnn_logits = cnn(x_train[:8])

assert digit_patches.shape == (8, 16, 4)
assert vit_logits.shape == cnn_logits.shape == (8, 10)
print({"patch sequence": tuple(digit_patches.shape),
       "CNN parameters": sum(p.numel() for p in cnn.parameters()),
       "ViT parameters": sum(p.numel() for p in vit.parameters())})
```

</details>

对 $8\times8$ 输入，十六个 $2\times2$ patch 让 tokenization 直接可见。严谨比较应匹配参数量、增强、搜索预算、预训练和计算量，并重复多个种子。该示例建立表示几何，而不是宣布优胜者。

### **本章对比与总结** {#chapter-comparison-summary}

CNN 设计是一系列关于**信息可以在哪里交互、分辨率如何变化、骨干网络向任务暴露什么接口**的决策。单独的层名称不如最终感受野、张量层级、计算模式和信息损失重要。

| 机制或家族 | 主要归纳偏置 | 效率优势 | 重要限制 |
|---|---|---|---|
| 图像全连接 MLP | 无显式空间局部性 | 实现简单 | 位置特定权重与较弱样本效率 |
| 标准卷积 | 局部共享模式与通道混合 | 固定核下随空间线性扩展 | 局部上下文只能靠深度/dilation 扩大 |
| stride/池化 | 更粗粒度空间摘要 | 降低激活内存与计算 | 混叠与不可逆细节损失 |
| 空洞卷积 | 稀疏的更宽上下文 | 不增加权重地扩大感受野 | 网格伪影且激活大小不变 |
| 深度可分离卷积 | 逐通道空间滤波后再混合 | 大幅减少参数/FLOPs | 交互受限且依赖硬件实现 |
| VGG 风格 stage | 重复小核 | 规则层级与多次非线性 | 激活开销高且无快捷路径 |
| 残差网络 | 围绕恒等映射学习变化 | 稳定深层优化 | 仍需正确形状对齐与归一化 |
| EfficientNet/MBConv | 平衡缩放与移动端分解 | 良好精度-效率权衡 | 配方和设备延迟都很重要 |
| ConvNeXt | 现代化卷积模块设计 | 可利用成熟高性能内核 | 大核与布局需要实测 |
| U-Net/FPN | 多分辨率融合 | 为密集任务恢复细节 | 保留和融合特征带来内存开销 |
| Vision Transformer | patch token 与内容相关全局混合 | 灵活的长距离交互 | 全局 attention 二次成本与较弱局部先验 |

共享 Digits 研究形成一条连续路线：

1. 保留 `[B,C,H,W]`，而不是通过展平丢掉邻域结构。
2. 把卷积理解为共享局部互相关，并验证精确索引。
3. 实现前先计算每个输出形状；stride 和 padding 会同时改变几何与成本。
4. 同时追踪感受野和特征分辨率，因为上下文与细节存在权衡。
5. 只在有明确效率或优化理由时使用分解、残差或现代化模块。
6. 根据输入分辨率调整架构规模，而不是照搬 ImageNet 拓扑名称。
7. 当输出具有空间结构时暴露多分辨率特征，并用空间指标评估任务。
8. 在匹配数据、配方、计算与部署条件下比较 CNN 和 Transformer。

本章代码中的每个示例都使用真实 digit 图像：手写互相关处理观测样本，感受野通过其像素探测，高效模块处理其特征图，编码器-解码器重建其前景。这种连续性让张量形状和架构假设变得具体。第 08 章将数据几何从二维空间改为有序时间，比较 recurrence、temporal convolution、attention 与 state-space model 在记忆和并行性上的不同权衡。